In [1]:
import pandas as pd
import numpy as np
import os

raw = "../data/raw/"
processed = "../data/processed/"
os.makedirs(processed, exist_ok=True)

# Load all needed tables
customers = pd.read_csv(raw + "olist_customers_dataset.csv")
orders    = pd.read_csv(raw + "olist_orders_dataset.csv")
items     = pd.read_csv(raw + "olist_order_items_dataset.csv")
products  = pd.read_csv(raw + "olist_products_dataset.csv")
payments  = pd.read_csv(raw + "olist_order_payments_dataset.csv")
reviews   = pd.read_csv(raw + "olist_order_reviews_dataset.csv")
sellers   = pd.read_csv(raw + "olist_sellers_dataset.csv")
geo       = pd.read_csv(raw + "olist_geolocation_dataset.csv")
trans     = pd.read_csv(raw + "product_category_name_translation.csv")

print("All tables loaded.")
for name, df in [("customers",customers),("orders",orders),("items",items),
                 ("products",products),("payments",payments),("reviews",reviews),
                 ("sellers",sellers),("geo",geo),("trans",trans)]:
    print(f"  {name:12s} {df.shape}")

All tables loaded.
  customers    (99441, 5)
  orders       (99441, 8)
  items        (112650, 7)
  products     (32951, 9)
  payments     (103886, 5)
  reviews      (99224, 7)
  sellers      (3095, 4)
  geo          (1000163, 5)
  trans        (71, 2)


In [2]:
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]
for c in date_cols:
    orders[c] = pd.to_datetime(orders[c], errors="coerce")

print(orders[date_cols].dtypes)
print("\nDate range:",
      orders["order_purchase_timestamp"].min(), "→",
      orders["order_purchase_timestamp"].max())

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

Date range: 2016-09-04 21:15:19 → 2018-10-17 17:30:18


In [3]:
null_report = orders.isnull().sum().to_frame("nulls")
null_report["pct"] = (null_report["nulls"] / len(orders) * 100).round(2)
print(null_report[null_report["nulls"] > 0])

                               nulls   pct
order_approved_at                160  0.16
order_delivered_carrier_date    1783  1.79
order_delivered_customer_date   2965  2.98


In [4]:
impossible = orders[
    (orders["order_delivered_customer_date"] < orders["order_purchase_timestamp"]) |
    (orders["order_approved_at"] < orders["order_purchase_timestamp"])
]
print(f"Impossible date rows: {len(impossible)}")
if len(impossible) > 0:
    print(impossible[["order_id","order_purchase_timestamp","order_approved_at",
                      "order_delivered_customer_date"]].head())

Impossible date rows: 0


In [5]:
print(orders["order_status"].value_counts())
print("\n% delivered:", round((orders["order_status"]=="delivered").mean()*100, 2))

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

% delivered: 97.02


In [6]:
print("Missing product_category_name:", products["product_category_name"].isnull().sum())
products["product_category_name"] = products["product_category_name"].fillna("unknown")

Missing product_category_name: 610


In [7]:
products = products.merge(trans, on="product_category_name", how="left")
products["product_category_name_english"] = (
    products["product_category_name_english"]
    .fillna(products["product_category_name"])  # keep portuguese if no translation
)
print("Translation coverage:",
      round((products["product_category_name_english"] != products["product_category_name"]).mean()*100, 2), "%")

Translation coverage: 91.9 %
